In [1]:
import pandas as pd
import re
from jiwer import wer
import ast
import Levenshtein
import numpy as np
import ast

# normalization library
import unicodedata
import contractions
from num2words import num2words

# google cloud library
from googleapiclient import discovery
from google.auth import default

/home/kelechi/miniconda3/envs/bio_ramp_env/lib/python3.9/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.21). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/home/kelechi/miniconda3/envs/bio_ramp_env/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/home/kelechi/miniconda3/envs/bio_ramp_env/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its e

In [2]:
# For Excel files, use read_excel instead
all_datasets = pd.read_excel("results/all_result_processed_normalized.xlsx", index_col=False)

In [12]:
def analyze_medical_text(project_id, location, text_content):
    """
    Call Google Healthcare API to analyze medical entities in text.
    Returns full API response payload for downstream mention reconstruction.
    """
    try:
        credentials, _ = default()
        service = discovery.build("healthcare", "v1", credentials=credentials)

        nlp_service_name = f"projects/{project_id}/locations/{location}/services/nlp"
        body = {"documentContent": text_content}

        response = service.projects().locations().services().nlp().analyzeEntities(
            nlpService=nlp_service_name,
            body=body,
        ).execute()

        return response
    except Exception as e:
        print(f"Error calling healthcare API: {e}")
        return {}


def _collect_mentions_from_response(response):
    """
    Extract mention spans in a normalized shape:
    {begin, end, replacement}
    """
    if not response:
        return []

    entities = response.get("entities", []) or []
    entity_mentions = response.get("entityMentions", []) or []

    # Map entity IDs to preferred terms from the entity catalog.
    id_to_term = {}
    for entity in entities:
        entity_id = entity.get("entityMention")
        preferred_term = entity.get("preferredTerm")
        if entity_id and preferred_term:
            id_to_term[entity_id] = preferred_term

    mentions_to_replace = []

    # Preferred path: top-level entityMentions usually contains reliable offsets.
    if entity_mentions:
        for mention in entity_mentions:
            text_obj = mention.get("text", {}) or {}
            surface_text = text_obj.get("content", "")
            begin_offset = text_obj.get("beginOffset")

            if begin_offset is None or not surface_text:
                continue

            linked_entities = mention.get("linkedEntities", []) or []
            linked_id = None
            if linked_entities:
                linked_id = linked_entities[0].get("entityMention")

            # Fall back to mention-level entityMention if linkedEntities is missing.
            entity_id = linked_id or mention.get("entityMention")
            if not entity_id:
                continue

            preferred_term = id_to_term.get(entity_id, surface_text)
            end_offset = begin_offset + len(surface_text)
            replacement_tag = f"[{entity_id}: {preferred_term}]"

            mentions_to_replace.append(
                {
                    "begin": begin_offset,
                    "end": end_offset,
                    "replacement": replacement_tag,
                }
            )

        return mentions_to_replace

    # Fallback path: some responses only include entity->mentions.
    for entity in entities:
        entity_id = entity.get("entityMention", "")
        preferred_term = entity.get("preferredTerm", "")
        if not entity_id or not preferred_term:
            continue

        mentions = entity.get("mentions", []) or []
        for mention in mentions:
            text_obj = mention.get("text", {}) or {}
            surface_text = text_obj.get("content", "")
            begin_offset = text_obj.get("beginOffset")

            if begin_offset is None or not surface_text:
                continue

            end_offset = begin_offset + len(surface_text)
            replacement_tag = f"[{entity_id}: {preferred_term}]"
            mentions_to_replace.append(
                {
                    "begin": begin_offset,
                    "end": end_offset,
                    "replacement": replacement_tag,
                }
            )

    return mentions_to_replace


def reconstruct_text_with_ner_tags(text_content, response):
    """
    Reconstruct the original text with inline NER tags.
    Replaces mention spans with format: [UMLS/C0000726: Abdomen]
    """
    if pd.isna(text_content) or text_content is None:
        return text_content

    text_content = str(text_content)
    mentions_to_replace = _collect_mentions_from_response(response)
    if not mentions_to_replace:
        return text_content

    # Sort by reverse offset to avoid shifting indices during replacement.
    mentions_to_replace.sort(key=lambda x: x["begin"], reverse=True)

    # Skip overlapping spans to prevent malformed replacements.
    reconstructed_text = text_content
    last_begin = len(text_content) + 1
    for mention in mentions_to_replace:
        begin = mention["begin"]
        end = mention["end"]
        replacement = mention["replacement"]

        if begin < 0 or end > len(reconstructed_text) or begin >= end:
            continue

        if end > last_begin:
            continue

        reconstructed_text = (
            reconstructed_text[:begin] + replacement + reconstructed_text[end:]
        )
        last_begin = begin

    return reconstructed_text


def apply_ner_to_row(text_content, project_id, location):
    """
    Apply NER analysis to one transcript row and return the tagged transcript.
    """
    if pd.isna(text_content) or text_content is None:
        return text_content

    text_content = str(text_content)
    if not text_content.strip():
        return text_content

    response = analyze_medical_text(project_id, location, text_content)
    return reconstruct_text_with_ner_tags(text_content, response)

In [11]:
# Apply NER processing to create norm_human_transcript_ner column
PROJECT_ID = "bio-ramp-ner"
LOCATION = "us-central1"

print(f"Processing {len(all_datasets)} rows for NER tagging...")
print("This may take a while depending on the number of rows and text length.\n")

# Use apply with lambda to process each row
all_datasets['norm_human_transcript_ner'] = all_datasets['norm_human_transcript'].apply(
    lambda text: apply_ner_to_row(text, PROJECT_ID, LOCATION)
)

print("\n✓ NER processing complete!")
print(f"Created column 'norm_human_transcript_ner' with {len(all_datasets)} rows")
print("\n" + "=" * 70)
print("Sample of first 3 rows:")
print("=" * 70)

# Display sample output
for idx in range(min(3, len(all_datasets))):
    print(f"\nRow {idx}:")
    print(f"Original:  {all_datasets['norm_human_transcript'].iloc[idx][:100]}...")
    print(f"NER Tagged: {all_datasets['norm_human_transcript_ner'].iloc[idx][:100]}...")

Processing 120 rows for NER tagging...
This may take a while depending on the number of rows and text length.



KeyboardInterrupt: 

In [13]:
# Quick validation on first 2 rows before full batch run
sample_df = all_datasets.head(2).copy()
sample_df["norm_human_transcript_ner"] = sample_df["norm_human_transcript"].apply(
    lambda text: apply_ner_to_row(text, PROJECT_ID, LOCATION)
)

for i, row in sample_df.iterrows():
    tagged_text = row["norm_human_transcript_ner"]
    has_tag = "[UMLS/" in str(tagged_text)
    print(f"Row {i} has inline UMLS tags: {has_tag}")
    print(str(tagged_text)[:220])
    print("-" * 80)

Row 0 has inline UMLS tags: False
good morning i am doctor smith from babylon can you just confirm your name date of birth and the first line of your address please hi my name is susan  thirty redbridge street sw two two hz hello and your date of birth f
--------------------------------------------------------------------------------
Row 1 has inline UMLS tags: False
 hello hi i am doctor jacob and welcome to babylon hi  hi so just before we start is it alright if you could confirm your name for me please yep  john doe okay and your date of birth  uhh  twentyone twelve and nineteen  
--------------------------------------------------------------------------------


In [10]:
all_datasets.to_excel('results/all_result_processed_normalized_with_ner_tagged.xlsx', index=False, engine='openpyxl')

# load the excel file and display the first few rows
df = pd.read_excel('results/all_result_processed_normalized_with_ner_tagged.xlsx', engine='openpyxl')

# move each models' results to separate sheets in the excel file
with pd.ExcelWriter('results/all_result_separate_sheets_normalized.xlsx', engine='openpyxl') as writer:
    whisper_cols = ['utterance_id', 'source', 'duration_sec', 'human-transcript', 'Whisper-ASR', 'norm_human_transcript', 'norm_human_transcript_ner', 'norm_whisper_asr', 
                    'norm_whisper_asr_wer', 'norm_whisper_asr_ins', 'norm_whisper_asr_del', 
                    'norm_whisper_asr_sub', 'whisper_aligned_df',
                    'norm_whisper_asr_Deletions', 'norm_whisper_asr_Insertions', 'norm_whisper_asr_Substitutions', 'whisper_reconstructed_ref']
    phi4_cols = ['utterance_id', 'source', 'duration_sec', 'human-transcript', 'Phi-4-ASR', 'norm_human_transcript', 'norm_human_transcript_ner', 'norm_phi4_asr', 
                 'norm_phi4_asr_wer', 'norm_phi4_asr_ins', 'norm_phi4_asr_del', 
                 'norm_phi4_asr_sub', 'phi4_aligned_df',
                 'norm_phi4_asr_Deletions', 'norm_phi4_asr_Insertions', 'norm_phi4_asr_Substitutions', 'phi4_reconstructed_ref']
    parakeet_cols = ['utterance_id', 'source', 'duration_sec', 'human-transcript', 'Nvidia-Parakeet-ASR', 'norm_human_transcript', 'norm_human_transcript_ner',  'norm_parakeet_asr', 
                     'norm_parakeet_wer', 'norm_parakeet_ins', 'norm_parakeet_del', 
                     'norm_parakeet_sub', 'parakeet_aligned_df',
                     'norm_parakeet_Deletions', 'norm_parakeet_Insertions', 'norm_parakeet_Substitutions', 'parakeet_reconstructed_ref']
    # granite_cols = ['utterance_id', 'source', 'duration_sec', 'human-transcript', 'IBM-Granite', 'norm_human_transcript', 'norm_granite', 
    #                 'norm_granite_wer', 'norm_granite_ins', 'norm_granite_del', 
    #                 'norm_granite_sub', 'granite_aligned_df',
    #                 'norm_granite_Deletions', 'norm_granite_Insertions', 'norm_granite_Substitutions', 'granite_reconstructed_ref']
    df_whisper = df.filter(items=whisper_cols, axis=1)
    df_phi4 = df.filter(items=phi4_cols, axis=1)
    df_parakeet = df.filter(items=parakeet_cols, axis=1)
    # df_granite = df.filter(items=granite_cols, axis=1)

    # Only write non-empty DataFrames to avoid invisible sheet error
    if not df_whisper.empty:
        df_whisper.to_excel(writer, sheet_name='Whisper-ASR Results', index=False)
    if not df_phi4.empty:
        df_phi4.to_excel(writer, sheet_name='Phi-4-ASR Results', index=False)
    if not df_parakeet.empty:
        df_parakeet.to_excel(writer, sheet_name='Nvidia-Parakeet-ASR Results', index=False)
    # if not df_granite.empty:
    #     df_granite.to_excel(writer, sheet_name='IBM-Granite Results', index=False)
        
    # resize each row in the sheets to 120px
    for sheet_name in ['Whisper-ASR Results', 'Phi-4-ASR Results', 'Nvidia-Parakeet-ASR Results']: #'Nvidia-Parakeet-ASR Results', 'IBM-Granite Results'
        worksheet = writer.sheets.get(sheet_name)
        if worksheet:
            for row_idx in range(1, len(df) + 2):  # +2 to account for header row and 1-based indexing
                worksheet.row_dimensions[row_idx].height = 120


In [9]:
# select three session with the following utterance ids: day4_consultation07, 1_Malaria, RES0073
selected_utterances = ['2_Diarrhea', '18_Pneumonia', '46aacf84-fdd1-490b-a857-633d2e7763a0_7d4de4c9d3488a4bbd35634cbd3a2b66_l1RjPEwA']
selected_data = all_datasets[all_datasets['utterance_id'].isin(selected_utterances)]

# make all columns lowercase for better readability
selected_data.columns = [col.lower() for col in selected_data.columns]

# rename "phi-4-asr" column to "phi4-asr" for better readability
selected_data.rename(columns={'phi-4-asr': 'phi4-asr'}, inplace=True)

# save the selected data to a new excel file
selected_data.to_excel('results/selected_sessions_normalized_with_ner_tagged.xlsx', index=False, engine='openpyxl')

/tmp/ipykernel_699555/3316763195.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_data.rename(columns={'phi-4-asr': 'phi4-asr'}, inplace=True)
